### Chain
In langchain , a chain is a pipeline that connects different components like prompts , LLMs , tools and output parser in a logical flow. Instead of calling the LLM directly every time you combine steps together into a structured workflow which makes it reusable , modular and easier to scale.
Each step process input and passes it to the next step util the final output is ready.

### Normal Chain
A normal chain is the simplest form of a chain.It consists of a PromptTemplate(Input formating), an LLM(model call) and OutputParser(to clean or structure the response).

In [7]:
from json import load
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
parser = StrOutputParser()

template = PromptTemplate.from_template("You are a helpful assistant. Answer the question: {question}")

chain = template | llm | parser

result = chain.invoke({"question": "What is the capital of France?"})
print(result)
# Normal chain Pipeline visualization
chain.get_graph().print_ascii()

The capital of France is **Paris**.
      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


### Sequential Chain

A sequential chain connects multiple steps one after another where the output of one step becomes the input of the next. Use case - generate a short summary based on selected sentiment.


In [ ]:
from json import load
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
parser = StrOutputParser()

seq_template1 = PromptTemplate.from_template("""
Give me a short summary on topic : {topic}""")

seq_template2 = PromptTemplate.from_template("""
Based on the summary, give me 5 key points on it \n {summary}""")

seq_chain = seq_template1 | llm | parser | seq_template2 | llm | parser # chain the two templates and the llm together
seq_chain_result = seq_chain.invoke({"topic": "Generative AI in short"})

print(seq_chain_result)


Here are 5 key points based on the summary:

1.  **Creates New Content:** Generative AI's primary function is to create new, original content, rather than just analyzing or classifying existing data.
2.  **Learns from Vast Data:** It learns patterns and structures by processing immense amounts of existing data (like text, images, audio, or code).
3.  **Generates Realistic Outputs:** Using its learned knowledge, it produces novel and realistic outputs.
4.  **Responds to Prompts:** These outputs are generated in response to simple human prompts.
5.  **Automates Creative Tasks:** It automates and augments creative tasks, thereby revolutionizing how digital content is produced and interacted with.


In [9]:
# To visualize the Sequence chain graph with all steps, we can use the `get_graph()` method and print it in ASCII format
seq_chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *      

### Parallel Chain
A parallel chain runs multiple chains at the same time on same input and then combines thier outputs into one structured result.

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
# (e.g., OPENAI_API_KEY)
load_dotenv()

# Initialize the LLM
# temperature=0.0 makes the output more deterministic
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# Converts AIMessage output into plain string
parser = StrOutputParser()

# Prompt Template 1
# Generates a short summary for the given topic
par_template1 = PromptTemplate.from_template("""
Give me a short summary on topic : {topic}""")

# Prompt Template 2
# Generates 5 key points for the given topic
#
# IMPORTANT:
# This prompt DOES NOT receive the summary generated
# by par_chain1.
#
# It receives the original input:
#
# {
#   "topic": "Generative AI"
# }
#
# because RunnableParallel sends the same input
# to all chains running in parallel.
par_template2 = PromptTemplate.from_template("""
Based on the summary, give me 5 key pints on it \n {topic}""")

# Chain 1 Flow:
#
# Input:
# {
#   "topic": "Generative AI"
# }
#
# -> PromptTemplate
# -> LLM
# -> String Parser
#
# Output:
# "Generative AI is a branch of AI..."
par_chain1 = par_template1 | llm | parser

# Chain 2 Flow:
#
# Input:
# {
#   "topic": "Generative AI"
# }
#
# -> PromptTemplate
# -> LLM
# -> String Parser
#
# Output:
# "1. Generates content..."
par_chain2 = par_template2 | llm | parser


# RunnableParallel executes multiple chains simultaneously.
#
# IMPORTANT:
# It sends the SAME INPUT to both chains.
#
# Input:
# {
#   "topic": "Generative AI"
# }
#
# goes to:
#
# par_chain1
# par_chain2
#
# at the same time.
#
# It DOES NOT pass the output of par_chain1
# to par_chain2.
#
# Output of RunnableParallel:
#
# {
#   "summary": "...",
#   "key_points": "..."
# }
parallel_chain = RunnableParallel(

  {
    "summary": par_chain1,
    "key_points": par_chain2
  }
)

# This template combines the outputs
# produced by RunnableParallel.
#
# Here:
#
# {summary}
#    -> output from par_chain1
#
# {key_points}
#    -> output from par_chain2
mer_template = PromptTemplate.from_template("""
  Combine the following information into a comprehensive output:
                                             
  Summary: {summary}
  Key Points: {key_points}

""")

# Merge Chain Flow:
#
# Dictionary Output from RunnableParallel
#             ->
#      PromptTemplate
#             ->
#           LLM
#             ->
#          Parser
#             ->
#     Final Combined Response
mer_chain = mer_template | llm | parser

# Complete Flow:
#
# Input:
# {
#   "topic": "Generative AI"
# }
#
#                  RunnableParallel
#                 /                \
#                /                  \
#               ▼                    ▼
#
#         par_chain1           par_chain2
#
#               ▼                    ▼
#
#      Summary Output       Key Points Output
#
#                \          /
#                 \        /
#                  ▼      ▼
#
# {
#   "summary": "...",
#   "key_points": "..."
# }
#
#                  ▼
#
#             mer_chain
#
#                  ▼
#
#      Final Combined Response
final_chain = parallel_chain | mer_chain

# Execute the complete pipeline
# and generate the final response
par_result = final_chain.invoke({"topic": "Generative AI"})

# Print final output
print(par_result)

## Generative AI: A Comprehensive Overview

Generative AI is a sophisticated type of artificial intelligence designed to **create new, original content** rather than merely analyzing or classifying existing data. It achieves this by learning intricate patterns, structures, and styles from vast datasets (such as text, images, audio, or video). This deep understanding then allows it to **generate novel outputs** that are often remarkably indistinguishable from human-created content.

### Core Capabilities and Outputs

The primary function of Generative AI models is to produce content that didn't exist before. This includes a wide range of outputs such as:
*   **Text:** Articles, code, stories, summaries, emails, and more.
*   **Images:** Realistic photos, artistic creations, design elements.
*   **Audio:** Music, speech, sound effects.
*   **Video:** Short clips, animations, deepfakes.
*   **Other:** 3D models, synthetic data.

### Impact and Diverse Applications

Generative AI is rapidl

In [23]:
# To visualize the final chain graph with all steps, we can use the `get_graph()` method and print it in ASCII format
final_chain.get_graph().print_ascii()

                +-----------------------------------+                  
                | Parallel<summary,key_points>Input |                  
                +-----------------------------------+                  
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati

### Conditional Chain
A conditional chain chooses which chain to execute based on a condition or rule. Its like an if-else statement of chain.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

class ReviewSentiment(BaseModel):

  # the sentiment of the movie review can be either positive, negative, or neutral
  # we use Literal to restrict the possible values for sentiment to these three options
  # using literal we can ensure that the output from the LLM is always one of these three values.
  sentiment: Literal["positve", "negative", "neutral"] = Field(description=" The sentiment of the movie review")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
str_parser = StrOutputParser()
pydantic_parser = PydanticOutputParser(pydantic_object=ReviewSentiment)

con_template1 = PromptTemplate.from_template("""
    Give me sentiment of the movie review {review} \n
    format_instruction: {format_instruction}
""",
partial_variables={"format_instruction": pydantic_parser.get_format_instructions()}
)

review_chain = con_template1 | llm | pydantic_parser

# print(review_chain.invoke({"review": "This movie was fantastic! The acting was superb and the plot was engaging."}))


pos_template = PromptTemplate.from_template("""
  The review is postive so write a short appreciation: \n {review}
""")

neg_template = PromptTemplate.from_template("""
  The review is negative so write a short critique: \n {review}
""")

neu_template = PromptTemplate.from_template("""
  The review is neutral so write a short balanced remarks: \n {review}
""")

defult_template = PromptTemplate.from_template("""
  I counln not detect the clear sentimen. Provide a netral response: \n {review}
""")


branch_chain = RunnableBranch(
  (lambda x: x.sentiment == "positive", pos_template | llm | str_parser),
  (lambda x: x.sentiment == "negative", neg_template | llm | str_parser),
  (lambda x: x.sentiment == "neutral", neu_template | llm | str_parser),
  defult_template | llm | str_parser
  )

final_con_chain = review_chain | branch_chain

review = "t was okay—some good moments, but nothing particularly memorable. Just an average watch."

print(review_chain.invoke({"review": review}))

final_con_result = final_con_chain.invoke({"review": review})

print(final_con_result)